In [ ]:
# ============================================================================
# NOVEL HYBRID BONE AGE ESTIMATION MODEL - COMPLETE FIXED VERSION
# ============================================================================
# All critical fixes applied:
# 1. ✅ Robust keypoint validation + fallback
# 2. ✅ Rough-age detach (first 5 epochs)
# 3. ✅ Reduced GAT heads (4→2, prevents overfitting)
# 4. ✅ Extended backbone freezing (epochs 8, 15)
# 5. ✅ Proper validation timing (after training)
# 6. ✅ Overfitting monitoring
# Expected: MAE 4.2-4.8 months (publishable if < 4.5)
# ============================================================================

!pip install torch_geometric

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool
import torchvision.models as models
import pandas as pd
import numpy as np
import cv2
from pathlib import Path
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import os

KEYPOINT_DIR = Path("/kaggle/input/gnn-keypoint-extraction/mediapipe_keypoints")
AGE_MEAN = 127.32
AGE_STD = 40.98

# ============================================================================
# MODEL COMPONENTS
# ============================================================================

class AgeAwareRegionalAttention(nn.Module):
    def __init__(self, num_regions=5, feature_dim=256):
        super().__init__()
        self.num_regions = num_regions
        self.age_encoder = nn.Sequential(
            nn.Linear(1, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU()
        )
        self.attention = nn.Sequential(
            nn.Linear(feature_dim + 128, 256), nn.ReLU(),
            nn.Linear(256, num_regions), nn.Softmax(dim=1)
        )
        
    def forward(self, region_features, age_estimate):
        B = region_features.size(0)
        age_feat = self.age_encoder(age_estimate)
        age_feat = age_feat.unsqueeze(1).expand(-1, self.num_regions, -1)
        combined = torch.cat([region_features, age_feat], dim=2)
        attention_weights = self.attention(combined.mean(dim=1))
        weighted_features = region_features * attention_weights.unsqueeze(2)
        return weighted_features, attention_weights

class MultiRegionFeatureExtractor(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.region_extractors = nn.ModuleList([
            self._build_region_extractor(pretrained) for _ in range(5)
        ])
        self.region_adapters = nn.ModuleList([
            nn.Sequential(
                nn.Linear(1280, 512), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(512, 256)
            ) for _ in range(5)
        ])
        
    def _build_region_extractor(self, pretrained):
        try:
            from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
            model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT if pretrained else None)
        except:
            model = models.mobilenet_v2(pretrained=pretrained)
        
        original_conv = model.features[0][0]
        new_conv = nn.Conv2d(1, original_conv.out_channels,
                           kernel_size=original_conv.kernel_size,
                           stride=original_conv.stride,
                           padding=original_conv.padding, bias=False)
        if pretrained:
            with torch.no_grad():
                new_conv.weight.data = original_conv.weight.data.mean(dim=1, keepdim=True)
        model.features[0][0] = new_conv
        return nn.Sequential(model.features, nn.AdaptiveAvgPool2d(1))
    
    def forward(self, region_images):
        region_features = []
        for i, (extractor, adapter) in enumerate(zip(self.region_extractors, self.region_adapters)):
            feat = extractor(region_images[i]).flatten(1)
            feat = adapter(feat)
            region_features.append(feat)
        return torch.stack(region_features, dim=1)

class RegionalRelationshipGNN(nn.Module):
    """OPTIMIZED: Reduced heads to prevent overfitting"""
    def __init__(self, input_dim=256, hidden_dim=256, num_layers=3):
        super().__init__()
        self.register_buffer('edge_index', self._build_region_graph())
        self.gnn_layers = nn.ModuleList()
        
        # Reduced heads: 2 instead of 4
        self.gnn_layers.append(GATConv(input_dim, hidden_dim, heads=2, concat=True, dropout=0.3))
        for _ in range(num_layers - 2):
            self.gnn_layers.append(GATConv(hidden_dim * 2, hidden_dim, heads=2, concat=True, dropout=0.3))
        self.gnn_layers.append(GATConv(hidden_dim * 2, hidden_dim, heads=1, concat=False, dropout=0.3))
    
    def _build_region_graph(self):
        edges = [
            [1,0],[0,1], [2,0],[0,2], [3,0],[0,3],  # Fingers to wrist
            [1,2],[2,1], [2,3],[3,2],  # Adjacent fingers
            [4,0],[0,4], [4,1],[1,4], [4,2],[2,4], [4,3],[3,4]  # Whole hand hub
        ]
        return torch.tensor(edges, dtype=torch.long).t()
    
    def forward(self, region_features):
        B, N, D = region_features.shape
        x = region_features.reshape(B * N, D)
        edge_index = self.edge_index.repeat(1, B)
        for i in range(B):
            edge_index[:, i*self.edge_index.shape[1]:(i+1)*self.edge_index.shape[1]] += i * N
        
        for layer in self.gnn_layers:
            x = F.elu(layer(x, edge_index))
        
        batch = torch.arange(B, device=region_features.device).repeat_interleave(N)
        return global_mean_pool(x, batch)

class CrossRegionAttentionFusion(nn.Module):
    def __init__(self, feature_dim=256, num_heads=4):
        super().__init__()
        self.multihead_attn = nn.MultiheadAttention(feature_dim, num_heads, dropout=0.2, batch_first=True)
        self.norm = nn.LayerNorm(feature_dim)
        
    def forward(self, region_features):
        attn_out, attn_weights = self.multihead_attn(region_features, region_features, region_features)
        x = self.norm(region_features + attn_out)
        return x, attn_weights

class NovelHybridBoneAgeNet(nn.Module):
    def __init__(self, use_gender=True):
        super().__init__()
        self.use_gender = use_gender
        self.region_extractor = MultiRegionFeatureExtractor(pretrained=True)
        self.cross_attention = CrossRegionAttentionFusion(feature_dim=256, num_heads=4)
        self.region_gnn = RegionalRelationshipGNN(input_dim=256, hidden_dim=256)
        
        self.rough_age_estimator = nn.Sequential(
            nn.Linear(256 * 5, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 1)
        )
        self.age_aware_attention = AgeAwareRegionalAttention(num_regions=5, feature_dim=256)
        
        if use_gender:
            self.gender_embed = nn.Embedding(2, 64)
            self.gender_fc = nn.Linear(64, 32)
        
        gender_dim = 32 if use_gender else 0
        total_dim = 256 + 256 + 256 + gender_dim
        
        self.final_fusion = nn.Sequential(
            nn.Linear(total_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
    
    def forward(self, region_images, gender=None, epoch=0):
        region_features = self.region_extractor(region_images)
        rough_age_normalized = self.rough_age_estimator(region_features.flatten(1))
        
        # CRITICAL FIX: Detach rough age during early training
        if epoch < 5:
            rough_age = (rough_age_normalized.detach() * AGE_STD + AGE_MEAN)
        else:
            rough_age = (rough_age_normalized * AGE_STD + AGE_MEAN)
        
        region_features_attn, _ = self.cross_attention(region_features)
        weighted_features, attention_weights = self.age_aware_attention(region_features_attn, rough_age)
        
        region_pooled = weighted_features.mean(dim=1)
        gnn_features = self.region_gnn(region_features_attn)
        weighted_pooled = weighted_features.sum(dim=1)
        
        combined = torch.cat([region_pooled, gnn_features, weighted_pooled], dim=1)
        
        if self.use_gender and gender is not None:
            gender_feat = F.relu(self.gender_fc(self.gender_embed(gender)))
            combined = torch.cat([combined, gender_feat], dim=1)
        
        bone_age_normalized = self.final_fusion(combined)
        bone_age = bone_age_normalized * AGE_STD + AGE_MEAN
        
        return bone_age.squeeze(-1), attention_weights, rough_age.squeeze(-1)

# ============================================================================
# DATASET WITH ROBUST KEYPOINT HANDLING
# ============================================================================

class MultiRegionBoneAgeDataset(torch.utils.data.Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self._filter_existing_images()

    def _resolve_image_path(self, img_id):
        for ext in ['.png', '.jpg', '.jpeg']:
            candidate = self.img_dir / "boneage-training-dataset" / f"{img_id}{ext}"
            if candidate.exists():
                return candidate
        return None

    def _filter_existing_images(self):
        valid_rows = []
        for _, row in self.df.iterrows():
            img_id = row['id']
            keypoint_path = KEYPOINT_DIR / f"{img_id}.npy"
            img_path = self._resolve_image_path(img_id)
            if img_path and keypoint_path.exists():
                valid_rows.append(row)
        
        missing = len(self.df) - len(valid_rows)
        self.df = pd.DataFrame(valid_rows).reset_index(drop=True)
        if missing > 0:
            print(f"⚠️ Skipped {missing} samples with missing data")
    
    def _extract_regions(self, img, keypoints):
        """CRITICAL FIX: Robust extraction with fallback"""
        h, w = img.shape
        regions = []
        fallback_count = 0
        
        def validate_keypoints(kp, img_shape):
            h, w = img_shape
            if np.any(kp[:, 0] < 0) or np.any(kp[:, 0] > w):
                return False
            if np.any(kp[:, 1] < 0) or np.any(kp[:, 1] > h):
                return False
            x_range = kp[:, 0].max() - kp[:, 0].min()
            y_range = kp[:, 1].max() - kp[:, 1].min()
            if x_range < w * 0.1 or y_range < h * 0.1:
                return False
            return True
        
        def safe_crop(img, y_min, y_max, x_min, x_max, min_size=40):
            y_min, y_max = max(0, int(y_min)), min(h, int(y_max))
            x_min, x_max = max(0, int(x_min)), min(w, int(x_max))
            if (y_max - y_min < min_size) or (x_max - x_min < min_size):
                return None
            cropped = img[y_min:y_max, x_min:x_max]
            if cropped.size == 0 or cropped.shape[0] < min_size or cropped.shape[1] < min_size:
                return None
            return cropped
        
        use_fallback = not validate_keypoints(keypoints, (h, w))
        if use_fallback:
            hand_img = img[h//8:7*h//8, w//8:7*w//8]
            return [hand_img]*5, 5
        
        # Extract each region with individual fallbacks
        region_configs = [
            (keypoints[:8], lambda kp: (kp.min(0), kp.max(0), 0.2, img[h//8:3*h//8, w//4:3*w//4])),
            (keypoints[[8,13,14]], lambda kp: (kp.min(0), kp.max(0), 0.3, img[2*h//8:5*h//8, 0:w//3])),
            (keypoints[[9,15,16]], lambda kp: (kp.min(0), kp.max(0), 0.3, img[h//8:5*h//8, w//3:2*w//3])),
            (keypoints[[10,17,18]], lambda kp: (kp.min(0), kp.max(0), 0.3, img[h//8:5*h//8, 2*w//3:w])),
            (keypoints, lambda kp: (kp.min(0), kp.max(0), 0.1, img[h//8:7*h//8, w//8:7*w//8]))
        ]
        
        for kp, config_fn in region_configs:
            (x_min, y_min), (x_max, y_max), margin_factor, fallback_img = config_fn(kp)
            margin = max(15, int(max(x_max-x_min, y_max-y_min) * margin_factor))
            region = safe_crop(img, y_min-margin, y_max+margin, x_min-margin, x_max+margin)
            if region is None:
                region = fallback_img
                fallback_count += 1
            regions.append(region)
        
        return regions, fallback_count

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id, bone_age, gender = row['id'], row['boneage'], row['male']
        
        keypoints = np.load(KEYPOINT_DIR / f"{img_id}.npy").astype(np.float32)
        img = cv2.imread(str(self._resolve_image_path(img_id)), cv2.IMREAD_GRAYSCALE)
        
        regions, fallback_count = self._extract_regions(img, keypoints)
        
        transformed_regions = []
        transform_simple = A.Compose([
            A.Resize(128, 128),
            A.Normalize(mean=[0.5], std=[0.25]),
            ToTensorV2()
        ])
        for region in regions:
            transformed_regions.append(transform_simple(image=region)['image'])
        
        bone_age_normalized = (bone_age - AGE_MEAN) / AGE_STD
        
        return {
            'regions': transformed_regions,
            'bone_age': torch.tensor(bone_age_normalized, dtype=torch.float32),
            'bone_age_original': torch.tensor(bone_age, dtype=torch.float32),
            'gender': torch.tensor(int(gender), dtype=torch.long),
            'img_id': img_id,
            'fallback_count': fallback_count
        }

# ============================================================================
# TRAINING UTILITIES (FIXED)
# ============================================================================

def collate_fn(batch):
    regions = [[] for _ in range(5)]
    bone_ages, bone_ages_original, genders, img_ids, fallback_counts = [], [], [], [], []
    
    for sample in batch:
        for i, region in enumerate(sample['regions']):
            regions[i].append(region)
        bone_ages.append(sample['bone_age'])
        bone_ages_original.append(sample['bone_age_original'])
        genders.append(sample['gender'])
        img_ids.append(sample['img_id'])
        fallback_counts.append(sample['fallback_count'])
    
    return {
        'regions': [torch.stack(r) for r in regions],
        'bone_age': torch.stack(bone_ages),
        'bone_age_original': torch.stack(bone_ages_original),
        'gender': torch.stack(genders),
        'img_id': img_ids,
        'fallback_count': fallback_counts
    }

def train_epoch(model, loader, optimizer, criterion, device, scaler=None, warmup_factor=1.0, epoch=0):
    model.train()
    total_loss = total_mae = total_fallbacks = 0
    
    pbar = tqdm(loader, desc='Training')
    for batch in pbar:
        regions = [r.to(device) for r in batch['regions']]
        target = batch['bone_age'].to(device)
        target_original = batch['bone_age_original'].to(device)
        gender = batch['gender'].to(device)
        
        total_fallbacks += sum(batch['fallback_count'])
        optimizer.zero_grad()
        
        if scaler:
            with torch.amp.autocast('cuda'):
                pred, _, _ = model(regions, gender, epoch=epoch)
                pred_normalized = (pred - AGE_MEAN) / AGE_STD
                loss = criterion(pred_normalized, target)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            if warmup_factor < 1.0:
                for pg in optimizer.param_groups:
                    pg['lr'] *= warmup_factor
            
            scaler.step(optimizer)
            scaler.update()
            
            if warmup_factor < 1.0:
                for pg in optimizer.param_groups:
                    pg['lr'] /= warmup_factor
        else:
            pred, _, _ = model(regions, gender, epoch=epoch)
            pred_normalized = (pred - AGE_MEAN) / AGE_STD
            loss = criterion(pred_normalized, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        mae = torch.abs(pred - target_original).mean()
        total_loss += loss.item()
        total_mae += mae.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'mae': f'{mae.item():.2f}'})
    
    fallback_rate = total_fallbacks / (len(loader.dataset) * 5) * 100 if len(loader.dataset) > 0 else 0
    if fallback_rate > 0:
        print(f"   ⚠️ Fallback rate: {fallback_rate:.1f}% of regions")
    
    return total_loss / len(loader), total_mae / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = total_mae = 0
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Validation'):
            regions = [r.to(device) for r in batch['regions']]
            target = batch['bone_age'].to(device)
            target_original = batch['bone_age_original'].to(device)
            gender = batch['gender'].to(device)
            
            with torch.amp.autocast('cuda'):
                pred, _, _ = model(regions, gender, epoch=999)  # High epoch = no detach
                pred_normalized = (pred - AGE_MEAN) / AGE_STD
                loss = criterion(pred_normalized, target)
            
            mae = torch.abs(pred - target_original).mean()
            total_loss += loss.item()
            total_mae += mae.item()
    
    return total_loss / len(loader), total_mae / len(loader)

def progressive_unfreeze(model, epoch, unfreeze_schedule):
    if epoch in unfreeze_schedule:
        layers_to_unfreeze = unfreeze_schedule[epoch]
        print(f"\n🔓 Unfreezing layers at epoch {epoch}: {layers_to_unfreeze}")
        
        for layer_name in layers_to_unfreeze:
            if layer_name == 'region_extractors':
                for param in model.region_extractor.parameters():
                    param.requires_grad = True
            elif layer_name == 'region_extractors_deep':
                for extractor in model.region_extractor.region_extractors:
                    for name, param in extractor.named_parameters():
                        if any(f'features.{i}' in name for i in [15,16,17,18]):
                            param.requires_grad = True
        
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())
        print(f"   Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

def freeze_backbone(model):
    print("🔒 Freezing region extractor backbones...")
    for extractor in model.region_extractor.region_extractors:
        for param in extractor.parameters():
            param.requires_grad = False
    for adapter in model.region_extractor.region_adapters:
        for param in adapter.parameters():
            param.requires_grad = True
    
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"   Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

def get_optimizer_with_unfrozen_params(model, config):
    backbone_params = []
    other_params = []
    
    for name, param in model.named_parameters():
        if param.requires_grad:
            if 'region_extractor.region_extractors' in name:
                backbone_params.append(param)
            else:
                other_params.append(param)
    
    if backbone_params:
        return torch.optim.AdamW([
            {'params': backbone_params, 'lr': config['lr'] * 0.1, 'weight_decay': 1e-4},
            {'params': other_params, 'lr': config['lr'], 'weight_decay': 1e-4}
        ])
    return torch.optim.AdamW([{'params': other_params, 'lr': config['lr'], 'weight_decay': 1e-4}])

def save_checkpoint(model, optimizer, scheduler, epoch, val_mae, config, checkpoint_dir='checkpoints', is_best=False):
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'val_mae': val_mae,
        'config': config
    }
    
    checkpoint_path = os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch}.pth')
    torch.save(checkpoint, checkpoint_path)
    
    if is_best:
        torch.save(checkpoint, os.path.join(checkpoint_dir, 'best_model.pth'))
        torch.save(checkpoint, os.path.join(checkpoint_dir, 'best_model.keras'))
        torch.save(model.state_dict(), os.path.join(checkpoint_dir, 'best_model_weights.pth'))
        torch.save({
            'model': model,
            'state_dict': model.state_dict(),
            'val_mae': val_mae,
            'epoch': epoch,
            'config': config
        }, os.path.join(checkpoint_dir, 'best_model_complete.pth'))
        
        print(f"✅ Saved best model (Epoch {epoch}, MAE: {val_mae:.2f} months)")

# ============================================================================
# MAIN TRAINING (FIXED)
# ============================================================================

def main():
    config = {
        'csv_path': '/kaggle/input/rsna-bone-age/boneage-training-dataset.csv',
        'img_dir': '/kaggle/input/rsna-bone-age/boneage-training-dataset',
        'batch_size': 16,
        'lr': 2e-4,
        'epochs': 60,
        'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        'use_amp': True,
        'use_gender': True,
        'warmup_epochs': 3
    }
    
    print("="*70)
    print("🚀 NOVEL HYBRID BONE AGE MODEL (ALL FIXES APPLIED)")
    print("="*70)
    print("✅ Robust keypoint validation")
    print("✅ Rough-age detach (first 5 epochs)")
    print("✅ Reduced GAT heads (2 instead of 4)")
    print("✅ Extended backbone freezing")
    print("✅ Proper validation timing")
    print("✅ Overfitting monitoring")
    print(f"Device: {config['device']}")
    print("="*70)
    
    # Load data
    df = pd.read_csv(config['csv_path'])
    train_df, val_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['male'])
    train_df.to_csv('train.csv', index=False)
    val_df.to_csv('val.csv', index=False)
    print(f"\nTrain: {len(train_df)} | Val: {len(val_df)}")
    
    train_dataset = MultiRegionBoneAgeDataset('train.csv', config['img_dir'])
    val_dataset = MultiRegionBoneAgeDataset('val.csv', config['img_dir'])
    
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'],
                             shuffle=True, num_workers=4, pin_memory=True,
                             persistent_workers=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'],
                           shuffle=False, num_workers=2, pin_memory=True,
                           persistent_workers=True, collate_fn=collate_fn)
    
    print("\n🧠 Building model...")
    model = NovelHybridBoneAgeNet(use_gender=config['use_gender']).to(config['device'])
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")
    
    # EXTENDED unfreezing schedule
    print("\n" + "="*70)
    print("🎯 PROGRESSIVE UNFREEZING (OPTIMIZED)")
    print("="*70)
    print("Phase 1 (Epochs 0-7): Fusion, GNN, attention (frozen backbone)")
    print("Phase 2 (Epochs 8-14): Deep backbone layers")
    print("Phase 3 (Epochs 15+): Full model")
    print("="*70)
    
    freeze_backbone(model)
    unfreeze_schedule = {
        8: ['region_extractors_deep'],
        15: ['region_extractors']
    }
    
    optimizer = get_optimizer_with_unfrozen_params(model, config)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=5, T_mult=2, eta_min=1e-6
    )
    
    criterion = nn.SmoothL1Loss()
    scaler = torch.amp.GradScaler('cuda') if config['use_amp'] else None
    
    best_mae = float('inf')
    history = {'train_loss': [], 'train_mae': [], 'val_loss': [], 'val_mae': []}
    patience = 0
    max_patience = 12
    
    print("\n" + "="*70)
    print("🏋️ Training...")
    print("="*70)
    
    for epoch in range(config['epochs']):
        print(f"\nEpoch {epoch+1}/{config['epochs']}")
        
        warmup_factor = 1.0
        if epoch < config['warmup_epochs']:
            warmup_factor = (epoch + 1) / config['warmup_epochs']
            print(f"🔥 Warmup: LR factor = {warmup_factor:.3f}")
        
        if epoch in unfreeze_schedule:
            progressive_unfreeze(model, epoch, unfreeze_schedule)
            optimizer = get_optimizer_with_unfrozen_params(model, config)
            remaining = config['epochs'] - epoch
            scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
                optimizer, T_0=max(5, remaining // 3), T_mult=2, eta_min=1e-6
            )
        
        # FIXED: Train first, then validate
        train_loss, train_mae = train_epoch(
            model, train_loader, optimizer, criterion, config['device'], scaler, warmup_factor, epoch
        )
        val_loss, val_mae = validate(model, val_loader, criterion, config['device'])
        
        if scheduler and epoch >= config['warmup_epochs']:
            scheduler.step()
        
        history['train_loss'].append(train_loss)
        history['train_mae'].append(train_mae)
        history['val_loss'].append(val_loss)
        history['val_mae'].append(val_mae)
        
        # OVERFITTING DETECTION
        gap = train_mae - val_mae
        if gap < -3.0:
            print(f"   ⚠️ Underfitting (gap: {gap:.2f} months)")
        elif gap > 2.0:
            print(f"   🚨 OVERFITTING (gap: {gap:.2f} months)")
        
        print(f"Train MAE: {train_mae:.2f} | Val MAE: {val_mae:.2f} months (gap: {gap:+.2f})")
        
        if val_mae < best_mae:
            best_mae = val_mae
            patience = 0
            save_checkpoint(model, optimizer, scheduler, epoch, val_mae, config, is_best=True)
        else:
            patience += 1
        
        if patience >= max_patience:
            print(f"\n⏹️ Early stopping at epoch {epoch+1}")
            break
    
    print(f"\n{'='*70}")
    print(f"🎯 Best Validation MAE: {best_mae:.2f} months")
    if best_mae < 4.5:
        print("✅ EXCELLENT RESULT - PUBLISHABLE!")
    elif best_mae < 5.0:
        print("✅ GOOD RESULT - Close to publishable")
    print(f"{'='*70}")
    
    # Plot results
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.plot(history['train_loss'], label='Train', linewidth=2, color='#3498db')
    plt.plot(history['val_loss'], label='Val', linewidth=2, color='#e74c3c')
    plt.xlabel('Epoch', fontweight='bold')
    plt.ylabel('Loss', fontweight='bold')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.title('Loss Curves', fontweight='bold')
    
    plt.subplot(1, 3, 2)
    plt.plot(history['train_mae'], label='Train', linewidth=2, color='#3498db')
    plt.plot(history['val_mae'], label='Val', linewidth=2, color='#e74c3c')
    plt.axhline(y=4.5, color='#27ae60', linestyle='--', linewidth=2, label='Target')
    plt.xlabel('Epoch', fontweight='bold')
    plt.ylabel('MAE (months)', fontweight='bold')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.title('MAE Curves', fontweight='bold')
    
    plt.subplot(1, 3, 3)
    epochs = range(1, len(history['val_mae']) + 1)
    plt.bar(epochs, history['val_mae'], color='#9b59b6', alpha=0.7)
    plt.axhline(y=best_mae, color='#e74c3c', linestyle='--', linewidth=2, 
                label=f'Best: {best_mae:.2f}')
    plt.xlabel('Epoch', fontweight='bold')
    plt.ylabel('Val MAE', fontweight='bold')
    plt.legend()
    plt.grid(alpha=0.3, axis='y')
    plt.title('Validation MAE', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    print("\n📊 Saved: training_curves.png")
    
    print("\n" + "="*70)
    print("📈 TRAINING SUMMARY")
    print("="*70)
    print(f"Total Epochs: {len(history['train_mae'])}")
    print(f"Best Epoch: {history['val_mae'].index(best_mae) + 1}")
    print(f"Final Train MAE: {history['train_mae'][-1]:.2f}")
    print(f"Final Val MAE: {history['val_mae'][-1]:.2f}")
    print(f"Best Val MAE: {best_mae:.2f}")
    print(f"Improvement vs baseline (5.0): {5.0 - best_mae:.2f} months")
    print("="*70)
    
    print("\n" + "="*70)
    print("🎓 NOVEL CONTRIBUTIONS")
    print("="*70)
    print("1. ✅ Multi-Region Feature Extraction")
    print("2. ✅ Age-Aware Regional Attention")
    print("3. ✅ Regional Relationship GNN")
    print("4. ✅ Cross-Region Attention Fusion")
    print("5. ✅ Robust Keypoint Handling")
    print("="*70)
    
    return model, history

if __name__ == '__main__':
    model, history = main()